# BioNOVA — Local End-to-End Demonstration

**Biomedical Literature Mining and Hypothesis Discovery**

This notebook walks through the complete BioNOVA pipeline step by step:

1. Environment check
2. Imports and configuration
3. Literature retrieval (with demo data fallback)
4. Data cleaning and normalization
5. Biomedical entity extraction
6. Relation extraction
7. Knowledge graph construction
8. Graph statistics
9. Temporal analysis
10. Gap candidate discovery
11. Novelty verification
12. Hypothesis generation
13. Hypothesis ranking
14. Supporting and contradicting evidence
15. Testable predictions
16. Export results

**No PubMed API key required** — the demo dataset is built-in and works offline.

## Cell 2 — Environment and Dependency Check

In [ ]:
import sys
import importlib
from pathlib import Path

# Ensure repo root is on path
_ROOT = Path('.').resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

print(f'Python {sys.version}')
print(f'Working directory: {_ROOT}')

REQUIRED = ['networkx', 'requests']
OPTIONAL = ['numpy', 'pandas', 'google.generativeai', 'spacy']

print('\nRequired packages:')
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg}  ← MISSING: pip install {pkg}')

print('\nOptional packages (for enhanced features):')
for pkg in OPTIONAL:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  - {pkg}  (not installed, fallback used)')

## Cell 3 — Imports

In [ ]:
import json
import dataclasses

from utils import setup_logging, BioNOVAConfig, normalize_text
from retrieval import (
    Article, XMLParser, DataCleaner,
    MockRetriever, create_sample_articles, load_local_articles,
)
from extraction import ExtractionPipeline, RuleBasedExtractor, Entity, Relation
from graph import BioNOVAGraph, build_graph, NoveltyStatus, GapCandidate
from hypothesis import (
    Hypothesis, HypothesisPipeline, generate_and_rank, EvidenceScorer,
)

logger = setup_logging('INFO')
print('All BioNOVA modules imported successfully.')

## Cell 4 — Configuration

In [ ]:
# Load configuration from environment / .env file (if present)
cfg = BioNOVAConfig.load_env()

# Display configuration
print('BioNOVA Configuration:')
print(f'  Entrez email:          {cfg.entrez_email or "(not set — using demo data)"}')
print(f'  Max results:           {cfg.max_results}')
print(f'  Gemini API key:        {"set" if cfg.gemini_api_key else "not set (rule-based generation)"}')
print(f'  Random seed:           {cfg.random_seed}')
print(f'  Historical cutoff:     {cfg.historical_cutoff_year or "none (use all literature)"}')

# For this demo, use built-in sample data (no API key needed)
USE_DEMO = not bool(cfg.entrez_email)
print(f'\nUsing demo data: {USE_DEMO}')

## Cell 5 — Input Biomedical Query

In [ ]:
# Configure query and parameters here
QUERY = 'BRCA1 breast cancer DNA repair'
MAX_RESULTS = 50
CUTOFF_YEAR = None  # Set to e.g. 2020 for temporal analysis

print(f'Query:              {QUERY}')
print(f'Max results:        {MAX_RESULTS}')
print(f'Historical cutoff:  {CUTOFF_YEAR or "none"}')

## Cell 6 — Retrieve Literature

In [ ]:
if USE_DEMO:
    print('Using built-in demo articles (no internet required).')
    raw_articles = create_sample_articles()
else:
    print(f'Searching PubMed for: {QUERY!r} ...')
    from retrieval import retrieve_and_clean
    try:
        raw_articles = retrieve_and_clean(
            query=QUERY,
            config=cfg,
            max_results=MAX_RESULTS,
        )
        if not raw_articles:
            print('No articles retrieved from PubMed. Falling back to demo data.')
            raw_articles = create_sample_articles()
    except Exception as e:
        print(f'PubMed retrieval failed: {e}. Falling back to demo data.')
        raw_articles = create_sample_articles()

print(f'\nRetrieved {len(raw_articles)} raw articles.')

## Cell 7 — Display Retrieved Publications

In [ ]:
print(f'First {min(5, len(raw_articles))} articles:\n')
for i, art in enumerate(raw_articles[:5]):
    print(f'[{i+1}] PMID: {art.pmid}')
    print(f'    Title: {art.title}')
    print(f'    Year: {art.year}  |  Journal: {art.journal}')
    if art.authors:
        print(f'    Authors: {art.authors[0]}{" et al." if len(art.authors) > 1 else ""}')
    print(f'    Abstract preview: {art.abstract[:120]}...')
    print()

## Cell 8 — Clean and Normalize Literature

In [ ]:
cleaner = DataCleaner()
articles = cleaner.clean_articles(raw_articles)

# Apply historical cutoff if set
if CUTOFF_YEAR:
    articles = [a for a in articles if a.year is None or a.year <= CUTOFF_YEAR]
    print(f'After cutoff ({CUTOFF_YEAR}): {len(articles)} articles')

removed = len(raw_articles) - len(articles)
print(f'Cleaned: {len(articles)} articles ({removed} duplicates/empty removed)')

# Publication year distribution
years = [a.year for a in articles if a.year]
if years:
    print(f'Publication years: {min(years)}–{max(years)}')

## Cell 9 — Extract Biomedical Entities

In [ ]:
extractor = ExtractionPipeline(use_scispacy=False)
entities, relations = extractor.process_articles(articles)

print(f'Extracted {len(entities)} entities')

# Entity type distribution
from collections import Counter
type_counts = Counter(e.entity_type for e in entities)
print('\nEntity types:')
for etype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {etype:30s} {count:4d}')

# Top entities by frequency
entity_freq = Counter(e.normalized for e in entities)
print('\nTop 10 entities:')
for ent, freq in entity_freq.most_common(10):
    print(f'  {ent:30s} {freq:3d} mentions')

## Cell 10 — Extract Biomedical Relations

In [ ]:
print(f'Extracted {len(relations)} relations\n')

# Polarity distribution
polarity_counts = Counter(r.polarity for r in relations)
print('Polarity distribution:')
for p, c in sorted(polarity_counts.items()):
    print(f'  {p:15s} {c}')

print('\nSample relations:')
for r in relations[:10]:
    neg_marker = '✗' if r.polarity == 'negative' else '??' if r.polarity in ('uncertain', 'conditional') else '✓'
    print(f'  {neg_marker} [{r.pmid}] {r.subject} --{r.predicate}--> {r.object} '
          f'(conf={r.confidence:.2f}, {r.certainty})')
    print(f'     "{r.sentence[:80]}..."' if len(r.sentence) > 80 else f'     "{r.sentence}"')

## Cell 11 — Build Evidence Knowledge Graph

In [ ]:
bg = build_graph(relations, cutoff_year=CUTOFF_YEAR)

print('Knowledge graph built.')
print(f'  Biological graph: {bg.graph.number_of_nodes()} nodes, '
      f'{bg.graph.number_of_edges()} edges')
print(f'  Semantic graph:   {bg.semantic_graph.number_of_nodes()} nodes, '
      f'{bg.semantic_graph.number_of_edges()} edges')
print()
print('NOTE: Biological and semantic edges are maintained in separate graphs.')
print('Semantic co-occurrence edges are never treated as biological evidence.')

## Cell 12 — Graph Statistics

In [ ]:
stats = bg.get_statistics()

print('Graph Statistics')
print('=' * 40)
print(f'Nodes:                   {stats["n_nodes"]}')
print(f'Biological edges:        {stats["n_biological_edges"]}')
print(f'Semantic edges:          {stats["n_semantic_edges"]}')
print(f'Graph density:           {stats["density"]:.4f}')
print(f'Average degree:          {stats["avg_degree"]:.2f}')
print(f'Isolated nodes:          {stats["isolated_nodes_count"]}')
print(f'Connected components:    {stats["connected_components"]}')
print()
print('Top entities by degree:')
for node, degree in stats['top_nodes_by_degree'][:10]:
    print(f'  {node:30s} degree={degree}')

## Cell 13 — Temporal Analysis

In [ ]:
# Collect publication years from relations
years_in_relations = sorted(set(r.publication_year for r in relations if r.publication_year))

if years_in_relations:
    print(f'Publication year range in extracted relations: '
          f'{min(years_in_relations)}–{max(years_in_relations)}')
    print()

    # Demonstrate temporal snapshot
    if len(years_in_relations) >= 2:
        mid_year = years_in_relations[len(years_in_relations) // 2]
        snapshot = bg.get_temporal_snapshot(mid_year)
        print(f'Temporal snapshot (cutoff year = {mid_year}):')
        print(f'  Nodes: {snapshot.number_of_nodes()}')
        print(f'  Edges: {snapshot.number_of_edges()}')
        print()
        print(f'Full graph (all years):')
        print(f'  Nodes: {bg.graph.number_of_nodes()}')
        print(f'  Edges: {bg.graph.number_of_edges()}')
        print()
        print(f'Edges added AFTER {mid_year}: '
              f'{bg.graph.number_of_edges() - snapshot.number_of_edges()}')
else:
    print('No publication year information available in this dataset.')

## Cell 14 — Gap Candidate Discovery

In [ ]:
gap_candidates = bg.find_gap_candidates(cutoff_year=CUTOFF_YEAR, top_k=20)

print(f'Found {len(gap_candidates)} gap candidates')
print()
print('Methods used: Jaccard similarity, Adamic–Adar index, Resource Allocation, '
      'Common Neighbors')
print('Filter: only biologically relevant entity type pairs '
      '(e.g. GENE↔DISEASE, CHEMICAL↔GENE)')
print()

if gap_candidates:
    print(f'{"Entity A":25s} {"Type A":12s} {"Entity B":25s} {"Type B":12s} '
          f'{"Score":8s} {"Method":18s} {"Shared Neighbors"}')
    print('-' * 120)
    for c in gap_candidates[:10]:
        shared = ', '.join(c.common_neighbors[:3])
        if len(c.common_neighbors) > 3:
            shared += f' (+{len(c.common_neighbors)-3})'
        print(f'{c.node_a:25s} {c.node_a_type:12s} {c.node_b:25s} {c.node_b_type:12s} '
              f'{c.score:8.4f} {c.method:18s} {shared}')
else:
    print('No gap candidates found. This can happen with small datasets where '
          'no two nodes share a common neighbor of compatible biological type.')

## Cell 15 — Novelty Verification

In [ ]:
from collections import Counter

print('Novelty Status Categories:')
print('  KNOWN                  — direct edge exists in the graph')
print('  PARTIALLY_SUPPORTED    — synonym/alias match or path length 2')
print('  INDIRECTLY_SUPPORTED   — indirect path (length 3)')
print('  UNEXPLORED             — no evidence found')
print('  CONTRADICTED           — negated evidence exists')
print('  INSUFFICIENT_EVIDENCE  — not enough evidence to classify')
print()
print('IMPORTANT: A candidate is not considered novel simply because')
print('the exact phrase does not appear in PubMed. Synonyms, aliases,')
print('and indirect evidence are explicitly checked.')
print()

if gap_candidates:
    status_counts = Counter(c.novelty_status.name for c in gap_candidates)
    print('Novelty status distribution:')
    for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
        print(f'  {status:30s} {count}')
else:
    print('No candidates to classify.')

## Cell 16 — Generate Hypotheses

In [ ]:
# Use LLM only if Gemini API key is set; otherwise use rule-based generation
USE_LLM = bool(cfg.gemini_api_key)
TOP_K_HYPOTHESES = 5

print(f'Generation mode: {"Gemini LLM" if USE_LLM else "Rule-based (deterministic)"}')
print(f'Processing top {TOP_K_HYPOTHESES} gap candidates...')
print()

pipeline = HypothesisPipeline(
    config_or_api_key=cfg.gemini_api_key if USE_LLM else None,
    use_llm=USE_LLM,
)

hypotheses = pipeline.generate_hypotheses(
    candidates=gap_candidates,
    relations=relations,
    query=QUERY,
    top_k=TOP_K_HYPOTHESES,
)

print(f'Generated {len(hypotheses)} hypotheses.')

## Cell 17 — Rank Hypotheses

In [ ]:
ranked = pipeline.rank_hypotheses(hypotheses)

print(f'Ranking {len(ranked)} hypotheses by final score')
print()
print('Scoring formula:')
print('  Final score = 0.35 × Evidence + 0.30 × Novelty + 0.25 × Plausibility + 0.10 × Feasibility')
print()
print(f'{"#":3s} {"Hypothesis (truncated)":50s} {"Final":6s} {"Evid":6s} '
      f'{"Novel":6s} {"Plaus":6s} {"Feasib":6s} {"Novelty Status"}')
print('-' * 130)

for i, h in enumerate(ranked):
    truncated = h.hypothesis[:48] + '..' if len(h.hypothesis) > 48 else h.hypothesis
    print(f'{i+1:3d} {truncated:50s} {h.final_score:6.3f} {h.evidence_score:6.3f} '
          f'{h.novelty_score:6.3f} {h.plausibility_score:6.3f} {h.feasibility_score:6.3f} '
          f'{h.novelty_status}')

## Cell 18 — Show Supporting and Contradicting Evidence

In [ ]:
if ranked:
    top_hyp = ranked[0]
    print(f'Top Hypothesis (ID: {top_hyp.hypothesis_id})')
    print('=' * 60)
    print(f'\nHypothesis: {top_hyp.hypothesis}')
    print(f'Entities: {top_hyp.entities}')
    print(f'Novelty status: {top_hyp.novelty_status}')
    print(f'Confidence: {top_hyp.confidence:.2f}')
    print()
    print('Mechanistic rationale:')
    print(f'  {top_hyp.mechanistic_rationale}')
    print()

    print(f'Supporting evidence ({len(top_hyp.supporting_evidence)} items):')
    for ev in top_hyp.supporting_evidence[:5]:
        pmid = ev.get('pmid', 'N/A')
        year = ev.get('year', 'N/A')
        sent = ev.get('sentence', '')[:120]
        print(f'  [PMID {pmid}, {year}] {sent}')

    print()
    print(f'Contradicting evidence ({len(top_hyp.contradicting_evidence)} items):')
    if top_hyp.contradicting_evidence:
        for ev in top_hyp.contradicting_evidence[:3]:
            pmid = ev.get('pmid', 'N/A')
            sent = ev.get('sentence', '')[:120]
            print(f'  [PMID {pmid}] {sent}')
    else:
        print('  (none found)')

    print()
    print("Devil's advocate:")
    print(f'  {top_hyp.devils_advocate}')
    print()
    print('Alternative explanations:')
    print(f'  {top_hyp.alternative_explanations}')
else:
    print('No hypotheses generated.')

## Cell 19 — Testable Predictions and Suggested Experiments

In [ ]:
for i, h in enumerate(ranked):
    print(f'Hypothesis {i+1}: {h.hypothesis[:60]}...')
    print(f'  Testable prediction:    {h.testable_prediction}')
    print(f'  Suggested experiment:   {h.suggested_experiment}')
    print(f'  Falsifiability:         {h.falsifiability}')
    print(f'  Limitations:            {h.limitations}')
    print()

## Cell 20 — Export Results

In [ ]:
import json
import dataclasses
from pathlib import Path

output_dir = Path('bionova_output')
output_dir.mkdir(exist_ok=True)

# Export articles
articles_out = [
    {'pmid': a.pmid, 'title': a.title, 'abstract': a.abstract,
     'year': a.year, 'journal': a.journal, 'authors': a.authors}
    for a in articles
]
(output_dir / 'articles.json').write_text(json.dumps(articles_out, indent=2))
print(f'Exported {len(articles_out)} articles → {output_dir}/articles.json')

# Export relations
rels_out = [dataclasses.asdict(r) for r in relations]
(output_dir / 'relations.json').write_text(json.dumps(rels_out, indent=2, default=str))
print(f'Exported {len(rels_out)} relations → {output_dir}/relations.json')

# Export gap candidates
gaps_out = [
    {
        'node_a': c.node_a, 'node_a_type': c.node_a_type,
        'node_b': c.node_b, 'node_b_type': c.node_b_type,
        'score': c.score, 'method': c.method,
        'common_neighbors': c.common_neighbors,
        'novelty_status': c.novelty_status.name,
    }
    for c in gap_candidates
]
(output_dir / 'gap_candidates.json').write_text(json.dumps(gaps_out, indent=2))
print(f'Exported {len(gaps_out)} gap candidates → {output_dir}/gap_candidates.json')

# Export hypotheses
if ranked:
    pipeline.export_hypotheses(ranked, str(output_dir / 'hypotheses.json'))
    print(f'Exported {len(ranked)} hypotheses → {output_dir}/hypotheses.json')

# Export analysis metadata
metadata = {
    'query': QUERY,
    'cutoff_year': CUTOFF_YEAR,
    'n_articles': len(articles),
    'n_entities': len(entities),
    'n_relations': len(relations),
    'n_nodes': stats['n_nodes'],
    'n_biological_edges': stats['n_biological_edges'],
    'n_gap_candidates': len(gap_candidates),
    'n_hypotheses': len(ranked),
    'use_llm': USE_LLM,
}
(output_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2))
print(f'Exported metadata → {output_dir}/metadata.json')

print('\nAnalysis Summary:')
for k, v in metadata.items():
    print(f'  {k:30s} {v}')